In this notebook, I will be making use of the Kaggle dataset conducted by a fictional e-commerce company, Shopster. This A/B Testing analysis is done in companion with resources from github and kaggle, as well as making use of youtube and google resources for better understanding. 


Through this realistic case study, we will explore from beginning to end of the A/B testing lifecycle:

1. Understanding of problem
2. Defining our hypothesis
3. Designing our experiment
4. Running the experiment
5. Validity checks
6. Analysing the results
7. Justifying launch decisions

---

Dataset

This dataset contains information on 20000 customers, who were subject to A/B testing for 31 days.

The columns in this dataset includes:
`customerID`: customer identifier
`test group`: subjecting customers into psa and ad groups
`made_purchase`: whether a user completed a purchase (binary)
`ad_count`: number of ads seen
`peak ad hours`: hours with highest ad exposure
`days_with_most_ads`: day with the most ad impressions

We will be making use of this dataset to walk through the entire A/B testing lifecycle, ultimately presenting our final business recommendations based on our testing.

---

1. Understanding of problem

The dataset is trying to explore whether a large company with a substantial user base plans to increase sales through advertisement on its website. The customers are tested through being segmented into 2 groups, `psa` and `ad`. 

Objective:

Through this test, the objective is to find out whether a redesigned ad format can increase the likelihood of customer purchases compared to traditional commercial versions. 

Looking at the user funnel, this intervention is targetting the purchase stage of the user funnel. Users were randomnly assigned to the control group (`psa`) and treatment group (`ad` ) group over a 31 day period.


Primary Metric

- *Conversion Rate*: percentage of users who completed a purchase (`made_purchase = True`)

Secondary Metric

- **Ad exposure volume**: number of ads seen per person
- **Peak Ad Hour**: Hours of highest ad exposure
- **Day of Peak Exposure**: Day of the month with most impressions

---

2. Define Hypothesis and parameters

Next, we will define the statistical framework of our experiment. We will be setting the parameters, formulatting the hypothesis and test type, in order to solidify our statistical thresholds and guide our inference of results.

Statistical parameters:

a. **Significance Level**: 0.05
b. **Statistical Power**: 0.80
This represents the probabilty of correctlty rejecting the null hypothesis when the alternative is true. A power of 80% reflects a common trade off between sensitivity and required sample size.
c. **Minimum Detectable Effect (MDE): 15%
This is the smallest relative change in the conversion rate that we consider practically meaningful from a business standpoint

Hypothesis Formulation:

**Null Hypothesis:** The conversion rate of the treatment group is less than or equal to 15% higher than that of the control group

**Alternative Hypothesis:** The conversion rate of the treatment group is more than 15% higher than the control group

Since we aim to detect whether the treatment leads to a conversion rate that is at least 15% higher than the control, we’ll conduct a one-sided test. 

In the next step, we’ll design the experiment.

---

3. Designing the experiment

In our dataset, each row represents a user that has been exposed to either the `psa` or the `ad` format. Hence, we assume that users were randomly assigned to either the control (`psa`) group or the treatment (`ad`) group at the beginning of the experiment

We will make use of the design parameters that we have chosen from the previous section (section 2):

**Primary Metric:** `made_purchase` used to identify Conversion Rate
**Secondary Metric:** `ad_count`: number of adds seen per user, `peak_ad_hours`: hour of hieghest ad exposure, `days_with_most_ads`: day of the month with the most impressions
**Significance Level:** 0.05
**Statistical Power:** 0.80
**Minimum Detectable Effect:** 15% relative increase in conversion
**Test Type:** One sided test

While many A/B tests make use of a 50/50 split, this experiment was designed with the intentional 60/40 allocation in favour of the treatment group. This allows for faster learning of the new ad's format and performance while still maintaining sufficient statistical power in the control group (40%).


What we know from historical data
Based on prior control group performance, the observed conversion rate `made_purchase` is:
- p_control = 0.0666, i.e 6.66% of people were converted
- variance: 0.0622, (p x (1-p)), where p is the probability of making a purchase and 1-p is the probability of not making a purchase


As we assume that the variance remains approximately equal across both control and treatment groups, we apply this value as the pooled variance in the sample size formula. 

With the MDE as 15%, the expected improvement threshold is:

**Δ = p_control × MDE = 0.0666 × 0.15 ≈ 0.00999** (about 1%)

henche yielding the below for the formula:

- σ² = 0.0622
- Δ² ≈ 0.0000998

Next, we want to determine how many people we need in our sample size in our control and treatment group for us to detect an improvement in conversion rate. 

For a one-sided test, we use the following formula:
$$
n \approx \frac{(Z_{1-\alpha} + Z_{1-\beta})^2 \cdot 2 \cdot \sigma^2}{\Delta^2}
$$

For a one-sided test at α = 0.05 and power = 0.80, we use:

Z₁₋α = 1.64
Z₁₋β = 0.84

$$
n \approx \frac{16 \cdot \sigma^2}{\Delta^2}
$$


---

4. Running the experiment

Key considerations include:

a. Random assignment: Users must be randomly assigned to the control or treatment group, and can only belong to one group

b. Consistent Exposure: Once the experiment begins, the logic in which ad is shown, formats, etc. should remain unchanged. 

c. Metric Instrumentation: All outcomes must be carefully logged in the way that they are presented to allow unbiased reporting

d. Avoiding peeking: Interim checks on results should be avoided to prevent bias points of views before the experiment has concluded

---

5. Validity checks

We need to perform data quality checks to ensure that the experiment was conducted under valid conditions. This can be in terms of 

- **Sample Ratio Mismatch:** Check whether the actual group allocation reflects the intended experimental desgin. A significant deviation in group sizes can indicate randomisation failure or data loss

- **Duration:** Ensuring that the experiment runs in a sufficient and stable timeline, with accounts for weekday effects, seasonality, potential novelty or maturation effects. 

- **Instrumentation Integrity:** Verify that all key variables are carefully populated and not missing values that can impede the experiment

- **Baseline Symmetry:** Compare pre-treatment characteristics (such as ad exposure volume or exposure timing) to confirm that groups were similar before any effect of the treatment could manifest.

- **Regression to the Mean:** Evaluate whether extreme baseline behaviors (e.g., high or low ad exposure) normalize across groups and time, potentially distorting the observed impact of the treatment.


As our dataset has been finalised, we will assume that the above points have been taken care of. 

Next, we will perform the statistical validation checks:


In [ ]:
import pandas as pd
from scipy.stats import chisquare 
import kagglehub #allows me to download and access kaggle datasets from python
import os

path = kagglehub.dataset_download(
    "farhadzeynalli/online-advertising-effectiveness-study-ab-testing"
)
print(os.listdir(path))

df = pd.read_csv(os.path.join(path, "online_ad_AB.csv"))

df.head()

['online_ad_AB.csv']


,customerID,test group,made_purchase,days_with_most_add,peak ad hours,ad_count
0,1,ad,False,24,20,5
1,2,psa,False,21,16,9
2,3,psa,False,1,18,8
3,4,ad,False,20,23,7
4,5,ad,False,3,13,5


Checking the column headings, we will replace the `days_with_most_add` to `days_with_most_ads`

In [5]:
df.rename(columns={'days_with_most_add': 'days_with_most_ads'}, inplace=True)
df.head()

,customerID,test group,made_purchase,days_with_most_ads,peak ad hours,ad_count
0,1,ad,False,24,20,5
1,2,psa,False,21,16,9
2,3,psa,False,1,18,8
3,4,ad,False,20,23,7
4,5,ad,False,3,13,5


In [6]:
# Check for missing values
missing_values = df.isnull().sum()

# Check data types
dtypes = df.dtypes

# Display results in a clean summary table
quality_check = pd.DataFrame({
    'Missing Values': missing_values,
    'Data Type': dtypes
})

display(quality_check)

,Missing Values,Data Type
customerID,0,int64
test group,0,object
made_purchase,0,bool
days_with_most_ads,0,int64
peak ad hours,0,int64
ad_count,0,int64


Next, we perform the relevant validity checks

Sample Mismatch


In [7]:
# Count users per group
group_counts = df['test group'].value_counts()
observed = group_counts.values

# Define expected proportions: 60% psa, 40% ad
expected_total = observed.sum()
expected = [expected_total * 0.6, expected_total * 0.4]

# Perform Chi-square test to detect Sample Ratio Mismatch (SRM)
chi2_stat, p_value = chisquare(f_obs=observed, f_exp=expected)

group_counts.to_dict(), expected, chi2_stat, p_value

print("Observed counts:", group_counts.to_dict())
print("Expected counts:", expected)
print("Chi-square statistic:", chi2_stat)
print("p-value:", p_value)

Observed counts: {'ad': 12053, 'psa': 7947}
Expected counts: [12000.0, 8000.0]
Chi-square statistic: 0.5852083333333333
p-value: 0.44427803130186905


Pre Treatment Symmetry

This is essential to ensure that groups are balanced in their pre-treatment characteristics.



In [8]:
# Compare pre-treatment variables between groups
grouped_summary = df.groupby('test group')[['ad_count', 'peak ad hours', 'days_with_most_ads']].agg(['mean', 'std'])

grouped_summary

ad_count           peak ad hours           days_with_most_ads  \
                mean       std          mean       std               mean   
test group                                                                  
ad          6.381067  2.554334     15.821870  6.229011          15.593960   
psa         6.408959  2.534506     15.964263  6.187886          15.448849   

                      
                 std  
test group            
ad          8.635782  
psa         8.690007

The differences is mean and standard deviation is minimal and not suggestive of imbalance